## Import and preprocessing

Import

In [255]:
%load_ext autoreload
%autoreload 2

from importlib import reload
import my_sklearn
reload(my_sklearn)

from my_sklearn import my_LinearRegression, my_Lasso, my_Ridge, my_ElasticNet, my_MinMaxScaler, my_StandardScaler

import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Read train and test json files, downloaded from [kaggle.com](https://www.kaggle.com/competitions/two-sigma-connect-rental-listing-inquiries/data?select=train.json.zip)

In [228]:
df = pd.read_json("data/train.json").reset_index()

df.head()

,index,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
0,4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
1,6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
2,9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
3,10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
4,15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [229]:
low = df['price'].quantile(0.01)
high = df['price'].quantile(0.99)
df = df[
    (df['price'] >= low) &
    (df['price'] <= high)
]

Generating additional features for better model quality

In [230]:
df['Features'] = df['features'].apply(
    lambda x: str(x)
    .replace('[', '')
    .replace(']', '')
    .replace('"', '')
    .replace("'", '')
    .replace(' ', '')
)

In [231]:
features = (
    df['Features']
    .str.split(',')
    .explode()
    .loc[lambda s: s != '']
    .tolist()
)
unique = set(features)
len(unique)

1529

In [232]:
top_feat = Counter(features).most_common(20)
top_feat

[('Elevator', 25398),
 ('HardwoodFloors', 23159),
 ('CatsAllowed', 23148),
 ('DogsAllowed', 21662),
 ('Doorman', 20497),
 ('Dishwasher', 20095),
 ('NoFee', 17806),
 ('LaundryinBuilding', 16093),
 ('FitnessCenter', 13000),
 ('Pre-War', 8978),
 ('LaundryinUnit', 8448),
 ('RoofDeck', 6423),
 ('OutdoorSpace', 5137),
 ('DiningRoom', 4901),
 ('HighSpeedInternet', 4225),
 ('Balcony', 2898),
 ('SwimmingPool', 2648),
 ('LaundryInBuilding', 2565),
 ('NewConstruction', 2507),
 ('Terrace', 2179)]

In [233]:
feature_list = [ftr[0] for ftr in top_feat]

X = pd.DataFrame()

for feature in feature_list:
    X[feature] = df['Features'].str.contains(feature, regex=False).astype('int8')

X['bathrooms'] = df['bathrooms']
X['bedrooms'] = df['bedrooms']

feature_list += ['bathrooms', 'bedrooms']

y = df['price']

## Models implementation

### Linear Regression

In [234]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=21,
    test_size=0.2
)

result_R2 = pd.DataFrame(columns=['model', 'train', 'test'])
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])

In [235]:
def my_R2(y, y_pred):

    y = np.asarray(y)
    y_pred = np.asarray(y_pred)

    if y.shape != y_pred.shape:
        raise ValueError('must be the same shape')

    y_mean = np.mean(y)
    tss = np.sum((y - y_mean)**2)
    
    if np.isclose(tss, 0):
        raise ValueError("TSS must be not 0")
    
    rss = np.sum((y - y_pred)**2)

    return 1 - rss / tss

def evaluate_model(model, name, result_MAE, result_RMSE, result_R2, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    MAE_test = mean_absolute_error(y_test, y_pred_test)
    MAE_train = mean_absolute_error(y_train, y_pred_train)

    RMSE_test = root_mean_squared_error(y_test, y_pred_test)
    RMSE_train = root_mean_squared_error(y_train, y_pred_train)

    R2_train = my_R2(y_train, y_pred_train)
    R2_test = my_R2(y_test, y_pred_test)

    result_MAE.loc[len(result_MAE)] = [name, MAE_train, MAE_test]
    result_RMSE.loc[len(result_RMSE)] = [name, RMSE_train, RMSE_test]
    result_R2.loc[len(result_R2)] = [name, R2_train, R2_test]

def results():
    print("result_MAE")
    display(result_MAE.sort_values(by='train'))

    print("result_RMSE")
    display(result_RMSE.sort_values(by='train'))

    print("result_R2")
    display(result_R2.sort_values(by='train', ascending=False))


#### sklearn linear regression

In [236]:
model = LinearRegression()
name = 'LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

#### my implementation

In [237]:
model = my_LinearRegression(random_state=21)
name = 'my_LinearRegression'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [238]:
model = my_LinearRegression(random_state=21, solver='gd')
name = 'my_LinearRegression_gd'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [239]:
model = my_LinearRegression(random_state=21, solver='analytical')
name = 'my_LinearRegression_analytical'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [240]:
results()

result_MAE


,model,train,test
3,my_LinearRegression_analytical,713.564590,708.139970
0,LinearRegression,713.564590,708.139970
1,my_LinearRegression,722.915332,716.743063
2,my_LinearRegression_gd,738.629522,728.100237


result_RMSE


,model,train,test
0,LinearRegression,1037.840266,1025.699099
3,my_LinearRegression_analytical,1037.840266,1025.699099
1,my_LinearRegression,1048.957051,1036.169793
2,my_LinearRegression_gd,1078.949643,1059.079576


result_R2


,model,train,test
0,LinearRegression,0.578442,0.586130
3,my_LinearRegression_analytical,0.578442,0.586130
1,my_LinearRegression,0.569363,0.577637
2,my_LinearRegression_gd,0.544384,0.558754


### Ridge, Lasso, ElasticNet

In [241]:
model = my_Ridge(random_state=21)
name = 'my_Ridge'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [242]:
model = my_Lasso(random_state=21)
name = 'my_Lasso'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [243]:
model = my_ElasticNet(random_state=21)
name = 'my_ElasticNet'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [244]:
model = Ridge()
name = 'Ridge'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [245]:
model = Lasso()
name = 'Lasso'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [246]:
model = ElasticNet()
name = 'ElasticNet'

evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    X_train, X_test, y_train, y_test
)

In [247]:
results()

result_MAE


,model,train,test
8,Lasso,713.068713,707.552171
7,Ridge,713.560603,708.132459
3,my_LinearRegression_analytical,713.564590,708.139970
0,LinearRegression,713.564590,708.139970
4,my_Ridge,722.906642,716.729994
6,my_ElasticNet,722.910980,716.736522
5,my_Lasso,722.915323,716.743051
1,my_LinearRegression,722.915332,716.743063
2,my_LinearRegression_gd,738.629522,728.100237
9,ElasticNet,810.098088,796.208102


result_RMSE


,model,train,test
0,LinearRegression,1037.840266,1025.699099
3,my_LinearRegression_analytical,1037.840266,1025.699099
7,Ridge,1037.840274,1025.695903
8,Lasso,1038.030037,1025.477172
1,my_LinearRegression,1048.957051,1036.169793
5,my_Lasso,1048.957056,1036.169792
6,my_ElasticNet,1048.960227,1036.171096
4,my_Ridge,1048.963402,1036.172406
2,my_LinearRegression_gd,1078.949643,1059.079576
9,ElasticNet,1193.587345,1177.106860


result_R2


,model,train,test
0,LinearRegression,0.578442,0.586130
3,my_LinearRegression_analytical,0.578442,0.586130
7,Ridge,0.578442,0.586133
8,Lasso,0.578288,0.586309
1,my_LinearRegression,0.569363,0.577637
5,my_Lasso,0.569363,0.577637
6,my_ElasticNet,0.569360,0.577636
4,my_Ridge,0.569358,0.577635
2,my_LinearRegression_gd,0.544384,0.558754
9,ElasticNet,0.442423,0.454926


### Feature normalization

Compare the feature normalization with your own method and with sklearn.

MinMaxScaler

In [248]:
my_scaler = my_MinMaxScaler()

my_scaler.fit(X_train)

my_X_train_scaled_minmax = my_scaler.transform(X_train)
my_X_test_scaled_minmax = my_scaler.transform(X_test)

scaler = MinMaxScaler()

scaler.fit(X_train)

X_train_scaled_minmax = my_scaler.transform(X_train)
X_test_scaled_minmax = my_scaler.transform(X_test)

Comparison Using `np.allclose`

`np.allclose()` is a NumPy function that checks whether two arrays are element-wise approximately equal within a specified tolerance.

Parameters:

- `a`, `b` — array-like inputs to compare.
- `rtol` — relative tolerance (default: $10^{-5}$).
- `atol` — absolute tolerance (default: $10^{-8}$).

Two values are considered equal if:

$$
|a - b| \leq atol + rtol \cdot |b|
$$

where:

- $a$ and $b$ are the elements being compared.
- $atol$ (Absolute Tolerance) is the minimum allowed tolerance. It is especially important for values close to zero.
- $rtol$ (Relative Tolerance) scales the tolerance according to the magnitude of the reference value $b$. It is useful when comparing larger numbers, where a fixed absolute tolerance may be too strict.

In [249]:
print(np.allclose(my_X_train_scaled_minmax, X_train_scaled_minmax))
print(np.allclose(my_X_test_scaled_minmax, X_test_scaled_minmax))

True
True


StandardScaler

In [250]:
my_scaler = my_StandardScaler()

my_scaler.fit(X_train)

my_X_train_scaled = my_scaler.transform(X_train)
my_X_test_scaled = my_scaler.transform(X_test)

scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = my_scaler.transform(X_train)
X_test_scaled = my_scaler.transform(X_test)

In [251]:
print(np.allclose(my_X_train_scaled, X_train_scaled))
print(np.allclose(my_X_test_scaled, X_test_scaled))

True
True


### Fit custom and sklearn models with normalized data

MinMaxScaler

In [252]:
models = {
    'my_LinearRegression_minmax' : my_LinearRegression(random_state=21),
    'my_Ridge_minmax' : my_Ridge(random_state=21),
    'my_Lasso_minmax' : my_Lasso(random_state=21),
    'my_ElasticNet_minmax' : my_ElasticNet(random_state=21),
    'LinearRegression_minmax' : LinearRegression(),
    'Ridge_minmax' : Ridge(),
    'Lasso_minmax' : Lasso(),
    'ElasticNet_minmax' : ElasticNet()
}

for name, model in models.items():
    evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    my_X_train_scaled_minmax, my_X_test_scaled_minmax, y_train, y_test
)

In [253]:
models = {
    'my_LinearRegression_scaled' : my_LinearRegression(random_state=21),
    'my_Ridge_scaled' : my_Ridge(random_state=21),
    'my_Lasso_scaled' : my_Lasso(random_state=21),
    'my_ElasticNet_scaled' : my_ElasticNet(random_state=21),
    'LinearRegression_scaled' : LinearRegression(),
    'Ridge_scaled' : Ridge(),
    'Lasso_scaled' : Lasso(),
    'ElasticNet_scaled' : ElasticNet()
}

for name, model in models.items():
    evaluate_model(
    model, name,
    result_MAE, result_RMSE, result_R2,
    my_X_train_scaled, my_X_test_scaled, y_train, y_test
)

In [254]:
results()

result_MAE


,model,train,test
12,my_Lasso_minmax,710.074333,704.858805
10,my_LinearRegression_minmax,710.074348,704.858827
13,my_ElasticNet_minmax,710.088185,704.785650
11,my_Ridge_minmax,710.119947,704.734149
8,Lasso,713.068713,707.552171
24,Lasso_scaled,713.342952,707.868396
16,Lasso_minmax,713.420667,707.495931
7,Ridge,713.560603,708.132459
23,Ridge_scaled,713.563868,708.138618
3,my_LinearRegression_analytical,713.564590,708.139970


result_RMSE


,model,train,test
0,LinearRegression,1037.840266,1025.699099
3,my_LinearRegression_analytical,1037.840266,1025.699099
14,LinearRegression_minmax,1037.840266,1025.699099
22,LinearRegression_scaled,1037.840266,1025.699099
23,Ridge_scaled,1037.840266,1025.698577
7,Ridge,1037.840274,1025.695903
24,Lasso_scaled,1037.869319,1025.559640
15,Ridge_minmax,1037.890962,1025.761675
8,Lasso,1038.030037,1025.477172
16,Lasso_minmax,1038.257181,1025.714200


result_R2


,model,train,test
22,LinearRegression_scaled,0.578442,0.586130
0,LinearRegression,0.578442,0.586130
14,LinearRegression_minmax,0.578442,0.586130
3,my_LinearRegression_analytical,0.578442,0.586130
23,Ridge_scaled,0.578442,0.586131
7,Ridge,0.578442,0.586133
24,Lasso_scaled,0.578419,0.586243
15,Ridge_minmax,0.578401,0.586080
8,Lasso,0.578288,0.586309
16,Lasso_minmax,0.578103,0.586118


### Overfit models

In [ ]:
poly = PolynomialFeatures(degree=10)

poly.fit(X_train[['bathrooms', 'bedrooms']])

X_train_poly = poly.transform()

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
45685,1,0,1,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,1,1.0,1
2928,0,0,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2.0,3
13374,1,1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,1,0,1.0,0
44390,0,1,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1.0,1
10971,0,0,1,1,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,2.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16768,0,0,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,2.0,3
9156,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
6072,1,1,0,0,1,1,1,1,1,1,...,1,0,1,0,0,0,0,0,2.0,3
5446,0,1,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,1


In [261]:
df['bedrooms']

0        1
1        2
2        2
3        3
4        0
        ..
49347    3
49348    2
49349    1
49350    2
49351    3
Name: bedrooms, Length: 48379, dtype: int64